# SignBridge v5 - AUTSL 226 Sinif Egitim (90%+ Hedef)

**Degisiklikler (v4'e gore):**
- Mixup KAPATILDI (train acc %27'de kaliyordu)
- Label smoothing 0.1 -> 0.02
- Augmentation hafiflettildi
- Velocity + Acceleration ozellikleri ACILDI (225 -> 675 feature)
- 3 fazli egitim: freeze -> partial -> full fine-tuning
- LR: 1e-5 (daha dusuk)
- Epochs: 100

**Kullanim:** Runtime > Run all > Drive izni ver > Bekle (~2 saat)

In [ ]:
#@title 1 - KURULUM + DRIVE
import subprocess, sys, os, shutil

for pkg in ['torch', 'torchvision', 'numpy', 'tqdm']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Paketler hazir')

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print('Drive bagli')

# Eski v5 checkpoint temizle
v5_dir = '/content/drive/MyDrive/AUTSL_Proje/Models_v5_pro'
if os.path.exists(v5_dir):
    for f in os.listdir(v5_dir):
        if f.endswith('.pt'):
            os.remove(os.path.join(v5_dir, f))
            print(f'Silindi: {f}')
os.makedirs(v5_dir, exist_ok=True)
print('Hazir')

In [ ]:
#@title 2 - KONFIGURASYON + VERI
import numpy as np, json, csv, math, time, random, gc
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class CONFIG:
    PROJE_DIR = '/content/drive/MyDrive/AUTSL_Proje'
    KOORDINAT_DIR = f'{PROJE_DIR}/Koordinatlar'
    CLASS_LIST_CSV = f'{PROJE_DIR}/SignList_ClassId_TR_EN (1).csv'
    SAVE_DIR = f'{PROJE_DIR}/Models_v5_pro'
    TRAIN_NPZ = f'{KOORDINAT_DIR}/all_train_packed.npz'
    VAL_NPZ = f'{KOORDINAT_DIR}/all_val_packed.npz'
    TEST_NPZ = f'{KOORDINAT_DIR}/all_test_packed.npz'
    # Model arch (MUST match pretrained)
    INPUT_SIZE = 225; D_MODEL = 384; NHEAD = 12; NUM_LAYERS = 6
    NUM_CLASSES = 226; DROPOUT = 0.3; SEQ_LENGTH = 30
    # v5 training params - LESS REGULARIZATION
    EPOCHS = 100; BATCH_SIZE = 64
    LR = 1e-5  # Dusuk LR
    WEIGHT_DECAY = 0.005
    WARMUP_EPOCHS = 5
    LABEL_SMOOTHING = 0.02  # Cok dusuk
    MIXUP_ALPHA = 0.0  # KAPALI
    GRADIENT_CLIP = 1.0; EMA_DECAY = 0.999
    # Feature engineering - ACIK
    USE_VELOCITY = True
    USE_ACCELERATION = True
    # Lighter augmentation
    AUG_NOISE_STD = 0.005  # Yarisina dusuruldu
    AUG_MIRROR_PROB = 0.2  # Dusuruldu
    AUG_SPEED_RANGE = (0.85, 1.15)  # Daraltildi
    AUG_SCALE_RANGE = (0.95, 1.05)  # Daraltildi
    AUG_DROPOUT_PROB = 0.05  # Dusuruldu
    PATIENCE = 25
    # 3-phase training
    PHASE1_EPOCHS = 10  # Sadece classifier egit
    PHASE2_EPOCHS = 30  # Son 3 transformer layer + classifier
    # Phase3: tum model (kalan epochlar)

cfg = CONFIG()

# Sinif isimleri
class_names = {}
if os.path.exists(cfg.CLASS_LIST_CSV):
    with open(cfg.CLASS_LIST_CSV, 'r', encoding='utf-8') as f:
        reader = csv.reader(f); next(reader, None)
        for row in reader:
            if len(row) >= 2:
                try: class_names[int(row[0])] = row[1].strip()
                except: pass
    print(f'{len(class_names)} sinif ismi yuklendi')

# Veri yukle
data = {}
for split, fpath in [('train', cfg.TRAIN_NPZ), ('val', cfg.VAL_NPZ), ('test', cfg.TEST_NPZ)]:
    if not os.path.exists(fpath):
        print(f'HATA {split} bulunamadi: {fpath}'); continue
    d = np.load(fpath); keys = list(d.keys())
    X = d['x'].astype(np.float32) if 'x' in keys else d[keys[0]].astype(np.float32)
    y = d['y'].astype(np.int64) if 'y' in keys else d[keys[1]].astype(np.int64)
    data[split] = (X, y)
    print(f'{split:5s}: {X.shape[0]:6d} ornek | shape={X.shape}')

X_train, y_train = data['train']
X_val, y_val = data['val']
X_test, y_test = data['test']
print('Veri yuklendi')

In [ ]:
#@title 3 - MODEL + DATASET + YARDIMCILAR

def augment_sequence(seq, cfg):
    aug = seq.copy()
    T = aug.shape[0]
    if random.random() < 0.4:
        new_T = max(10, int(T * random.uniform(*cfg.AUG_SPEED_RANGE)))
        idx = np.clip(np.linspace(0, T-1, new_T).astype(int), 0, T-1)
        aug = aug[idx]
    if random.random() < 0.2:
        aug = np.roll(aug, random.randint(-2, 2), axis=0)
    if random.random() < 0.4:
        aug = aug + np.random.randn(*aug.shape).astype(np.float32) * cfg.AUG_NOISE_STD
    if random.random() < cfg.AUG_MIRROR_PROB:
        m = aug.copy()
        for i in range(0, min(225, aug.shape[1]), 3): m[:, i] = 1.0 - m[:, i]
        if aug.shape[1] >= 225:
            lh, rh = m[:, 99:162].copy(), m[:, 162:225].copy()
            m[:, 99:162], m[:, 162:225] = rh, lh
        aug = m
    if random.random() < 0.2:
        s = random.uniform(*cfg.AUG_SCALE_RANGE)
        for i in range(0, min(225, aug.shape[1]), 3): aug[:, i] *= s; aug[:, i+1] *= s
    if random.random() < cfg.AUG_DROPOUT_PROB:
        n_lm = min(75, aug.shape[1] // 3)
        for li in random.sample(range(n_lm), max(1, int(n_lm * 0.05))):
            aug[:, li*3:li*3+3] = 0.0
    return aug

def build_features(sequence, cfg):
    features = [sequence]
    if cfg.USE_VELOCITY:
        v = np.zeros_like(sequence)
        v[1:] = sequence[1:] - sequence[:-1]
        features.append(v)
    if cfg.USE_ACCELERATION:
        a = np.zeros_like(sequence)
        a[2:] = sequence[2:] - 2*sequence[1:-1] + sequence[:-2]
        features.append(a)
    return np.concatenate(features, axis=-1).astype(np.float32)

def pad_or_truncate(sequence, target_length):
    T = len(sequence)
    if T == target_length: return sequence
    elif T > target_length:
        s = (T - target_length) // 2; return sequence[s:s + target_length]
    else:
        return np.concatenate([sequence, np.tile(sequence[-1:], (target_length - T, 1))], axis=0)

def compute_feature_size(cfg):
    size = cfg.INPUT_SIZE
    if cfg.USE_VELOCITY: size += cfg.INPUT_SIZE
    if cfg.USE_ACCELERATION: size += cfg.INPUT_SIZE
    return size

class AUTSLDataset(Dataset):
    def __init__(self, X, y, cfg, is_train=True):
        self.X, self.y, self.cfg, self.is_train = X, y, cfg, is_train
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        seq = self.X[idx].copy()
        if self.is_train: seq = augment_sequence(seq, self.cfg)
        seq = pad_or_truncate(seq, self.cfg.SEQ_LENGTH)
        seq = build_features(seq, self.cfg)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# ===== MODEL =====
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SignTransformerPro(nn.Module):
    def __init__(self, input_size, d_model=384, nhead=12, num_layers=6, num_classes=226, dropout=0.35):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Linear(input_size, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.conv_block = nn.Sequential(
            nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model),
            nn.Conv1d(d_model, d_model, 1),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model//4), nn.Tanh(), nn.Linear(d_model//4, 1))
            for _ in range(4)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model*5), nn.Dropout(dropout),
            nn.Linear(d_model*5, d_model*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(d_model, num_classes))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.input_conv(x)
        x = x + self.conv_block(x.transpose(1,2)).transpose(1,2)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        x = self.transformer(self.pos_encoder(x))
        seq = x[:, 1:]
        pooled = [F.softmax(h(seq), dim=1) * seq for h in self.pool_heads]
        pooled = [p.sum(dim=1) for p in pooled]
        return self.classifier(torch.cat(pooled + [seq.mean(dim=1)], dim=1))

class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad: self.shadow[name] = param.data.clone()
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    def apply(self, model):
        backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                backup[name] = param.data.clone(); param.data = self.shadow[name]
        return backup
    def restore(self, model, backup):
        for name, param in model.named_parameters():
            if param.requires_grad and name in backup: param.data = backup[name]
    def state_dict(self): return dict(self.shadow)
    def load_state_dict(self, sd):
        self.shadow = {k: v.clone() for k, v in sd.items() if k in self.shadow}

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-7):
        self.optimizer, self.warmup = optimizer, warmup_epochs
        self.total, self.min_lr = total_epochs, min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
    def step(self, epoch):
        if epoch < self.warmup:
            p = epoch / max(1, self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs): pg['lr'] = blr * p
        else:
            p = (epoch - self.warmup) / max(1, self.total - self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = self.min_lr + (blr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * p))
    def update_base_lr(self, new_lr):
        self.base_lrs = [new_lr] * len(self.base_lrs)

print('Model + Dataset + Augmentation hazir')

In [ ]:
#@title 4 - 3 FAZLI EGITIM

feature_size = compute_feature_size(cfg)
print(f'Feature size: {feature_size}')

# Normalization
print('Normalization hesaplaniyor...')
all_features = []
for i in range(0, len(X_train), 500):
    batch = X_train[i:i+500]
    for seq in batch:
        seq_p = pad_or_truncate(seq, cfg.SEQ_LENGTH)
        feat = build_features(seq_p, cfg)
        all_features.append(feat)
all_features = np.concatenate(all_features, axis=0)
norm_mean = all_features.mean(axis=0)
norm_std = np.maximum(all_features.std(axis=0), 1e-6)
print(f'  norm shape: {norm_mean.shape}')
del all_features; gc.collect()

# Datasets
train_dataset = AUTSLDataset(X_train, y_train, cfg, is_train=True)
val_dataset = AUTSLDataset(X_val, y_val, cfg, is_train=False)
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
norm_mean_t = torch.tensor(norm_mean, dtype=torch.float32).to(device)
norm_std_t = torch.tensor(norm_std, dtype=torch.float32).to(device)
print(f'Device: {device}')

# Model - input_size=675 (vel+acc acik) ama pretrained 225 ile egitildi
# Bu yuzden input_conv'u yeniden olusturacagiz
model = SignTransformerPro(
    input_size=feature_size,
    d_model=cfg.D_MODEL, nhead=cfg.NHEAD,
    num_layers=cfg.NUM_LAYERS, num_classes=cfg.NUM_CLASSES,
    dropout=cfg.DROPOUT
).to(device)

# Load pretrained weights (partial - skip input_conv since size changed)
pretrained_path = os.path.join(cfg.PROJE_DIR, 'Models_v3_80plus', 'best_model.pt')
if os.path.exists(pretrained_path):
    state = torch.load(pretrained_path, map_location=device, weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        pretrained_sd = state['model_state_dict']
        print(f'Pretrained val_acc: {state.get("val_acc", "?")}')
    else:
        pretrained_sd = state
    
    # Partial load - skip mismatched layers
    model_sd = model.state_dict()
    loaded, skipped = 0, 0
    for k, v in pretrained_sd.items():
        if k in model_sd and model_sd[k].shape == v.shape:
            model_sd[k] = v
            loaded += 1
        else:
            skipped += 1
    model.load_state_dict(model_sd)
    print(f'Pretrained: {loaded} katman yuklendi, {skipped} atlanildi (boyut uyumsuz)')
else:
    print(f'UYARI: Pretrained bulunamadi: {pretrained_path}')

total_params = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre: {total_params:,}')

# ===== FREEZE HELPER =====
def freeze_all(model):
    for p in model.parameters(): p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def unfreeze_classifier(model):
    freeze_all(model)
    for p in model.classifier.parameters(): p.requires_grad = True
    for p in model.pool_heads.parameters(): p.requires_grad = True
    for p in model.input_conv.parameters(): p.requires_grad = True  # input_conv yeni, egitmek lazim
    model.cls_token.requires_grad = True

def unfreeze_partial(model):
    """Son 3 transformer layer + classifier + input_conv"""
    freeze_all(model)
    for p in model.classifier.parameters(): p.requires_grad = True
    for p in model.pool_heads.parameters(): p.requires_grad = True
    for p in model.input_conv.parameters(): p.requires_grad = True
    for p in model.conv_block.parameters(): p.requires_grad = True
    model.cls_token.requires_grad = True
    for p in model.pos_encoder.parameters(): p.requires_grad = True
    # Son 3 transformer layer
    num_layers = len(model.transformer.layers)
    for i in range(max(0, num_layers - 3), num_layers):
        for p in model.transformer.layers[i].parameters(): p.requires_grad = True

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ===== VALIDATION FUNCTION =====
def validate(model, val_loader, device, norm_mean_t, norm_std_t):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx = (bx.to(device) - norm_mean_t) / norm_std_t
            by = by.to(device)
            with torch.amp.autocast('cuda'):
                out = model(bx)
            correct += (out.argmax(1) == by).sum().item()
            total += by.size(0)
    return 100 * correct / total

# ===== TRAIN ONE EPOCH =====
def train_epoch(model, train_loader, optimizer, criterion, scaler, device, norm_mean_t, norm_std_t, epoch, total_epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{total_epochs}')
    for bx, by in pbar:
        bx = (bx.to(device) - norm_mean_t) / norm_std_t
        by = by.to(device)
        with torch.amp.autocast('cuda'):
            out = model(bx)
            loss = criterion(out, by)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * by.size(0)
        correct += (out.argmax(1) == by).sum().item()
        total += by.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100*correct/total:.1f}%')
    return total_loss / total, 100 * correct / total

# ===== MAIN TRAINING =====
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
scaler = torch.amp.GradScaler('cuda')

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'lr': [], 'phase': []}
global_epoch = 0

# Check for existing checkpoint
checkpoint_path = os.path.join(cfg.SAVE_DIR, 'checkpoint.pt')
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    ckpt_acc = ckpt.get('val_acc', 0)
    if ckpt_acc >= 50.0:
        model.load_state_dict(ckpt['model_state_dict'])
        global_epoch = ckpt['epoch'] + 1
        best_val_acc = ckpt.get('best_val_acc', ckpt_acc)
        if 'history' in ckpt: history = ckpt['history']
        print(f'Checkpoint yuklendi: epoch {global_epoch}, val_acc={best_val_acc:.2f}%')
    else:
        os.remove(checkpoint_path)
        print(f'Bozuk checkpoint silindi (acc={ckpt_acc:.1f}%)')

# ========== PHASE 1: Classifier + input_conv only ==========
phase1_start = global_epoch
phase1_end = cfg.PHASE1_EPOCHS

if global_epoch < phase1_end:
    print(f'\n{"="*60}')
    print(f'FAZ 1: Classifier + input_conv egitimi (epoch {phase1_start+1}-{phase1_end})')
    print(f'{"="*60}')
    
    unfreeze_classifier(model)
    print(f'Egitilecek parametre: {count_trainable(model):,}')
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR * 5,  # Yeni katmanlar icin daha yuksek LR
        weight_decay=cfg.WEIGHT_DECAY
    )
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    
    for epoch in range(phase1_start, phase1_end):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS)
        ema.update(model)
        
        ema_backup = ema.apply(model)
        val_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        ema.restore(model, ema_backup)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['phase'].append(1)
        
        print(f'  [FAZ1] train_acc={train_acc:.1f}%, val_acc={val_acc:.2f}%')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            ema_b = ema.apply(model)
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': val_acc, 'epoch': epoch}, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            ema.restore(model, ema_b)
            print(f'  Yeni best: {val_acc:.2f}%')
        
        global_epoch = epoch + 1

# ========== PHASE 2: Partial unfreeze ==========
phase2_end = cfg.PHASE1_EPOCHS + cfg.PHASE2_EPOCHS

if global_epoch < phase2_end:
    print(f'\n{"="*60}')
    print(f'FAZ 2: Son 3 transformer layer + classifier (epoch {global_epoch+1}-{phase2_end})')
    print(f'{"="*60}')
    
    unfreeze_partial(model)
    print(f'Egitilecek parametre: {count_trainable(model):,}')
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR * 2,
        weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = WarmupCosineScheduler(optimizer, 3, phase2_end - global_epoch)
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    patience = 0
    
    for epoch in range(global_epoch, phase2_end):
        rel_epoch = epoch - global_epoch
        scheduler.step(rel_epoch)
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS)
        ema.update(model)
        
        ema_backup = ema.apply(model)
        val_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        ema.restore(model, ema_backup)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['phase'].append(2)
        
        print(f'  [FAZ2] train_acc={train_acc:.1f}%, val_acc={val_acc:.2f}%, lr={optimizer.param_groups[0]["lr"]:.2e}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience = 0
            ema_b = ema.apply(model)
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': val_acc, 'epoch': epoch}, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            ema.restore(model, ema_b)
            print(f'  Yeni best: {val_acc:.2f}%')
        else:
            patience += 1
        
        if (epoch + 1) % 5 == 0:
            torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'epoch': epoch, 'val_acc': val_acc, 'best_val_acc': best_val_acc, 'history': history}, checkpoint_path)
            print(f'  Checkpoint kaydedildi')
        
        global_epoch = epoch + 1

# ========== PHASE 3: Full fine-tuning ==========
if global_epoch < cfg.EPOCHS:
    print(f'\n{"="*60}')
    print(f'FAZ 3: Tum model fine-tuning (epoch {global_epoch+1}-{cfg.EPOCHS})')
    print(f'{"="*60}')
    
    unfreeze_all(model)
    print(f'Egitilecek parametre: {count_trainable(model):,}')
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )
    remaining = cfg.EPOCHS - global_epoch
    scheduler = WarmupCosineScheduler(optimizer, 2, remaining)
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    patience = 0
    
    for epoch in range(global_epoch, cfg.EPOCHS):
        rel_epoch = epoch - global_epoch
        scheduler.step(rel_epoch)
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS)
        ema.update(model)
        
        ema_backup = ema.apply(model)
        val_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        ema.restore(model, ema_backup)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['phase'].append(3)
        
        print(f'  [FAZ3] train_acc={train_acc:.1f}%, val_acc={val_acc:.2f}%, lr={optimizer.param_groups[0]["lr"]:.2e}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience = 0
            ema_b = ema.apply(model)
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': val_acc, 'epoch': epoch}, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            ema.restore(model, ema_b)
            print(f'  Yeni best: {val_acc:.2f}%')
        else:
            patience += 1
        
        if (epoch + 1) % 5 == 0:
            torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'epoch': epoch, 'val_acc': val_acc, 'best_val_acc': best_val_acc, 'history': history}, checkpoint_path)
            print(f'  Checkpoint kaydedildi')
        
        if patience >= cfg.PATIENCE:
            print(f'  Early stopping ({cfg.PATIENCE} epoch iyilesme yok)')
            break
        
        global_epoch = epoch + 1

print(f'\nEgitim tamamlandi! Best val_acc: {best_val_acc:.2f}%')

In [ ]:
#@title 5 - TEST + ANALIZ + KAYIT

print('='*60)
print('TEST DEGERLENDIRME')
print('='*60)

# Load best
best_path = os.path.join(cfg.SAVE_DIR, 'best_model.pt')
if os.path.exists(best_path):
    best_state = torch.load(best_path, map_location=device, weights_only=True)
    if 'model_state_dict' in best_state:
        model.load_state_dict(best_state['model_state_dict'])
    else:
        model.load_state_dict(best_state)
    print(f'Best model yuklendi (val_acc: {best_state.get("val_acc", "?")})')

test_dataset = AUTSLDataset(X_test, y_test, cfg, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for bx, by in tqdm(test_loader, desc='Test'):
        bx = (bx.to(device) - norm_mean_t) / norm_std_t
        by = by.to(device)
        with torch.amp.autocast('cuda'):
            out = model(bx)
        probs = F.softmax(out, dim=1)
        preds = out.argmax(1)
        test_correct += (preds == by).sum().item()
        test_total += by.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(by.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = 100 * test_correct / test_total
print(f'\nTest Accuracy: {test_acc:.2f}% ({test_correct}/{test_total})')

# Save normalization + label map
np.save(os.path.join(cfg.SAVE_DIR, 'norm_mean.npy'), norm_mean)
np.save(os.path.join(cfg.SAVE_DIR, 'norm_std.npy'), norm_std)
label_map = {str(i): class_names.get(i, f'class_{i}') for i in range(cfg.NUM_CLASSES)}
with open(os.path.join(cfg.SAVE_DIR, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print('norm + label_map kaydedildi')

# Training curves
if len(history['val_acc']) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs_range = range(1, len(history['val_acc'])+1)
    
    # Color by phase
    colors = ['blue' if p==1 else 'green' if p==2 else 'red' for p in history.get('phase', [3]*len(history['val_acc']))]
    
    axes[0].plot(epochs_range, history['train_loss'], 'b-', linewidth=2)
    axes[0].set_title('Training Loss'); axes[0].grid(True)
    
    axes[1].plot(epochs_range, history['val_acc'], 'r-', linewidth=2)
    if 'train_acc' in history:
        axes[1].plot(epochs_range, history['train_acc'], 'b--', linewidth=1, alpha=0.5, label='Train')
    axes[1].axhline(y=best_val_acc, color='g', linestyle='--', label=f'Best: {best_val_acc:.2f}%')
    axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True)
    
    axes[2].plot(epochs_range, history['lr'], 'g-', linewidth=2)
    axes[2].set_title('Learning Rate'); axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=150)
    plt.show()

# Per-class analysis
all_preds_np = np.array(all_preds)
all_labels_np = np.array(all_labels)
all_probs_np = np.array(all_probs)

per_class_correct = {}
per_class_total = {}
for p, l in zip(all_preds_np, all_labels_np):
    per_class_total[l] = per_class_total.get(l, 0) + 1
    if p == l: per_class_correct[l] = per_class_correct.get(l, 0) + 1

worst = []
for cid in per_class_total:
    acc = 100 * per_class_correct.get(cid, 0) / per_class_total[cid]
    name = class_names.get(cid, f'class_{cid}')
    worst.append((cid, name, acc, per_class_total[cid]))
worst.sort(key=lambda x: x[2])

print(f'\nEn dusuk 15 sinif:')
for cid, name, acc, n in worst[:15]:
    print(f'  {cid:>4} {name:<30} {acc:>6.1f}% (n={n})')

# Top-5
all_probs_t = torch.tensor(all_probs_np)
all_labels_t = torch.tensor(all_labels_np)
top5 = all_probs_t.topk(5, dim=1).indices
top5_correct = sum(1 for i in range(len(all_labels_t)) if all_labels_t[i] in top5[i])
top5_acc = 100 * top5_correct / len(all_labels_t)

correct_mask = all_preds_np == all_labels_np
correct_conf = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][correct_mask]
wrong_conf = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][~correct_mask]

print(f'\nOZET:')
print(f'  Test Top-1: {test_acc:.2f}%')
print(f'  Test Top-5: {top5_acc:.2f}%')
print(f'  Best Val:   {best_val_acc:.2f}%')
print(f'  Dogru conf: {correct_conf.mean():.4f}')
if len(wrong_conf) > 0:
    print(f'  Yanlis conf: {wrong_conf.mean():.4f}')
print(f'  Epochs:     {len(history["val_acc"])}')

print(f'\nKaydedilen dosyalar ({cfg.SAVE_DIR}):')
for f in sorted(os.listdir(cfg.SAVE_DIR)):
    fp = os.path.join(cfg.SAVE_DIR, f)
    sz = os.path.getsize(fp) / (1024*1024)
    print(f'  {f} ({sz:.1f} MB)')
print('\nTamamlandi!')